# v9 Supervised Counterfactual Ensemble Ranker

This self-contained notebook trains the v9 supervised ensemble ranker from `candidates_v7.csv`. It embeds the trainer code directly, uses the previous winning training method scaled for the 2500-game both-sides dataset, and uploads artifacts to `devaanshpa/orbit-wars-agent/v9`.


In [ ]:
from getpass import getpass
import os

if not os.environ.get("HF_TOKEN"):
    token = getpass("HF_TOKEN: " ).strip()
    if not token:
        raise RuntimeError("HF_TOKEN is required to download data and upload v9 artifacts")
    os.environ["HF_TOKEN"] = token

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))


In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd()
if not (repo_root / "notebooks" / "v9" / "train_v9_ranker.py").exists() and (repo_root.parent.parent / "notebooks" / "v9" / "train_v9_ranker.py").exists():
    repo_root = repo_root.parent.parent

data_path = os.environ.get("CANDIDATES_CSV", "").strip()
prefer_local = os.environ.get("V9_PREFER_LOCAL_DATA", "0") == "1"
if data_path:
    print("Using explicit CANDIDATES_CSV:", data_path)
elif prefer_local:
    local = sorted((repo_root / "data").glob("*/candidates_v7.csv"), key=lambda p: p.stat().st_mtime, reverse=True) if (repo_root / "data").exists() else []
    if local:
        os.environ["CANDIDATES_CSV"] = str(local[0])
        print("Using newest local CSV because V9_PREFER_LOCAL_DATA=1:", local[0])
    else:
        print("No local CSV found. The trainer will download the newest Hugging Face candidates_v7.csv.")
else:
    print("Default data source: newest Hugging Face data/*/candidates_v7.csv. Set CANDIDATES_CSV or V9_PREFER_LOCAL_DATA=1 to override.")


In [ ]:
import importlib.util
import subprocess
import sys

missing = []
for package, module in (("torch", "torch"), ("huggingface-hub", "huggingface_hub"), ("matplotlib", "matplotlib")):
    if importlib.util.find_spec(module) is None:
        missing.append(package)
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda devices:", torch.cuda.device_count())


In [ ]:
import sys
import types

V9_RANKER_CODE = 'import argparse\nimport csv\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\n\nHF_REPO_ID = "devaanshpa/orbit-wars-agent"\nHF_REPO_TYPE = "model"\nHF_REMOTE_PREFIX = "v9"\n\nMETADATA_COLS = {\n    "label",\n    "selected",\n    "outcome_weight",\n    "game_result",\n    "reward_margin",\n    "agent_reward",\n    "opponent_reward",\n    "selected_heuristic_rank",\n    "counterfactual_positive",\n    "counterfactual_reason",\n    "failure_overcommit",\n    "failure_missed_tactical",\n    "failure_missed_comet",\n    "failure_slow_expansion",\n    "turn_advantage",\n    "future_advantage_delta_5",\n    "future_advantage_delta_15",\n    "future_advantage_delta_30",\n    "future_production_delta_15",\n    "future_planet_delta_15",\n    "game_id",\n    "candidate_id",\n    "version",\n}\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description="Train the v9 Orbit Wars scaled supervised ensemble ranker.")\n    parser.add_argument("--csv", default=os.environ.get("CANDIDATES_CSV", ""))\n    parser.add_argument(\n        "--prefer-local-data",\n        action="store_true",\n        help="Use newest local data/*/candidates_v7.csv before Hugging Face. Default is to prefer Hugging Face.",\n    )\n    parser.add_argument("--export-dir", default="notebooks/v9/exports")\n    parser.add_argument("--epochs", type=int, default=int(os.environ.get("V9_EPOCHS", "280")))\n    parser.add_argument("--batch-size", type=int, default=int(os.environ.get("V9_BATCH_SIZE", "4096")))\n    parser.add_argument("--lr", type=float, default=float(os.environ.get("V9_LR", "0.00045")))\n    parser.add_argument("--weight-decay", type=float, default=float(os.environ.get("V9_WEIGHT_DECAY", "0.00030")))\n    parser.add_argument("--pair-loss-weight", type=float, default=float(os.environ.get("V9_PAIR_LOSS_WEIGHT", "1.05")))\n    parser.add_argument("--max-pairs-per-turn", type=int, default=int(os.environ.get("V9_MAX_PAIRS_PER_TURN", "12")))\n    parser.add_argument("--ensemble-size", type=int, default=int(os.environ.get("V9_ENSEMBLE_SIZE", "8")))\n    parser.add_argument("--patience", type=int, default=int(os.environ.get("V9_PATIENCE", "36")))\n    parser.add_argument("--dropout", type=float, default=float(os.environ.get("V9_DROPOUT", "0.13")))\n    parser.add_argument("--score-scale", type=float, default=float(os.environ.get("V9_SCORE_SCALE", "205.0")))\n    parser.add_argument("--seed", type=int, default=int(os.environ.get("V9_SEED", "907")))\n    parser.add_argument("--checkpoint-every", type=int, default=int(os.environ.get("V9_CHECKPOINT_EVERY", "40")))\n    parser.add_argument("--upload", action="store_true")\n    parser.add_argument("--hf-repo-id", default=HF_REPO_ID)\n    parser.add_argument("--hf-repo-type", default=HF_REPO_TYPE)\n    return parser.parse_args()\n\n\ndef load_dotenv(path=".env"):\n    env_path = Path(path)\n    if not env_path.exists():\n        return\n    for raw_line in env_path.read_text(encoding="utf-8").splitlines():\n        line = raw_line.strip()\n        if not line or line.startswith("#") or "=" not in line:\n            continue\n        key, value = line.split("=", 1)\n        key = key.strip()\n        value = value.strip().strip("\\"\'")\n        if key and key not in os.environ:\n            os.environ[key] = value\n\n\ndef find_training_csv(csv_arg, prefer_local=False):\n    if csv_arg:\n        path = Path(csv_arg)\n        if not path.exists():\n            raise FileNotFoundError(f"Training CSV does not exist: {path}")\n        return path\n\n    fixed = Path("notebooks/v9/data/candidates_v7.csv")\n    if prefer_local and fixed.exists():\n        return fixed\n\n    root = Path("data")\n    candidates = sorted(root.glob("*/candidates_v7.csv"), key=lambda path: path.stat().st_mtime, reverse=True) if root.exists() else []\n    if prefer_local and candidates:\n        return candidates[0]\n\n    load_dotenv()\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if not token:\n        raise FileNotFoundError("No candidates_v7.csv found locally and HF_TOKEN is not set for download.")\n    try:\n        from huggingface_hub import HfApi, hf_hub_download\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to download data: pip install huggingface_hub") from exc\n\n    api = HfApi(token=token)\n    files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)\n    remote_csvs = sorted(\n        [name for name in files if name.startswith("data/") and name.endswith("/candidates_v7.csv")],\n        reverse=True,\n    )\n    if not remote_csvs:\n        raise FileNotFoundError("No candidates_v7.csv found locally or in Hugging Face data folders.")\n    return Path(hf_hub_download(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, filename=remote_csvs[0], token=token))\n\n\ndef row_float(row, key, default=0.0):\n    try:\n        return float(row.get(key, default) or default)\n    except (TypeError, ValueError):\n        return float(default)\n\n\ndef read_rows(path):\n    with Path(path).open(newline="", encoding="utf-8") as f:\n        rows = list(csv.DictReader(f))\n    if not rows:\n        raise RuntimeError("Training CSV has no rows.")\n    feature_names = [name for name in rows[0] if name not in METADATA_COLS]\n    x_raw = [[row_float(row, name, 0.0) for name in feature_names] for row in rows]\n    y = [max(0.0, min(1.0, row_float(row, "label", 0.0))) for row in rows]\n    sample_weights = [max(0.05, row_float(row, "outcome_weight", 1.0)) for row in rows]\n    selected = [row_float(row, "selected", 0.0) >= 0.5 for row in rows]\n    counterfactual = [row_float(row, "counterfactual_positive", 0.0) >= 0.5 for row in rows]\n    return rows, feature_names, x_raw, y, sample_weights, selected, counterfactual\n\n\ndef split_by_game(rows, seed, validation_fraction=0.2):\n    games = sorted({row.get("game_id", "") for row in rows})\n    rng = random.Random(seed)\n    rng.shuffle(games)\n    valid_game_count = max(1, int(len(games) * validation_fraction)) if len(games) > 1 else 1\n    valid_games = set(games[:valid_game_count])\n    valid_indices = [i for i, row in enumerate(rows) if row.get("game_id", "") in valid_games]\n    valid_set = set(valid_indices)\n    train_indices = [i for i in range(len(rows)) if i not in valid_set] or valid_indices[:]\n    return train_indices, valid_indices, games, valid_games\n\n\ndef normalize_from_train(x_raw, train_indices):\n    train_raw = [x_raw[i] for i in train_indices]\n    means = [sum(col) / len(col) for col in zip(*train_raw)]\n    scales = []\n    for j, mean in enumerate(means):\n        var = sum((row[j] - mean) ** 2 for row in train_raw) / max(1, len(train_raw) - 1)\n        scales.append(max(1e-6, math.sqrt(var)))\n\n    def normalize(items):\n        return [[(row[j] - means[j]) / scales[j] for j in range(len(means))] for row in items]\n\n    return means, scales, normalize\n\n\ndef build_pair_indices(rows, labels, selected, counterfactual, indices, local_index, max_pairs_per_turn):\n    grouped = defaultdict(list)\n    for raw_index in indices:\n        step = int(row_float(rows[raw_index], "step", 0.0))\n        grouped[(rows[raw_index].get("game_id", ""), step)].append(raw_index)\n\n    pairs = []\n    for raw_indices in grouped.values():\n        positives = [\n            i\n            for i in raw_indices\n            if selected[i] or counterfactual[i] or labels[i] >= 0.55\n        ]\n        negatives = [\n            i\n            for i in raw_indices\n            if labels[i] <= 0.30 and not counterfactual[i]\n        ]\n        if not positives or not negatives:\n            continue\n        positives.sort(key=lambda i: labels[i] + row_float(rows[i], "outcome_weight", 1.0) * 0.05, reverse=True)\n        negatives.sort(key=lambda i: row_float(rows[i], "heuristic_score_scaled", 0.0), reverse=True)\n        for pos in positives:\n            for neg in negatives[:max_pairs_per_turn]:\n                if labels[pos] <= labels[neg] + 0.12:\n                    continue\n                gap = max(0.05, labels[pos] - labels[neg])\n                margin_weight = 1.0 + min(1.5, abs(row_float(rows[pos], "reward_margin", 0.0)) / 600.0)\n                hard_pair_weight = 1.35 if abs(row_float(rows[pos], "heuristic_score_scaled", 0.0) - row_float(rows[neg], "heuristic_score_scaled", 0.0)) <= 0.08 else 1.0\n                weight = gap * row_float(rows[pos], "outcome_weight", 1.0) * margin_weight * hard_pair_weight\n                if counterfactual[pos]:\n                    weight *= 1.45\n                pairs.append((local_index[pos], local_index[neg], max(0.05, weight)))\n    return pairs\n\n\ndef sigmoid_prob(value):\n    value = max(-50.0, min(50.0, value))\n    return 1.0 / (1.0 + math.exp(-value))\n\n\ndef grouped_top1_rate(rows, predictions, positive_mask, indices):\n    grouped = defaultdict(list)\n    for raw_index in indices:\n        step = int(row_float(rows[raw_index], "step", 0.0))\n        grouped[(rows[raw_index].get("game_id", ""), step)].append(raw_index)\n\n    hits = 0\n    total = 0\n    rank_fractions = []\n    for raw_indices in grouped.values():\n        positives = [i for i in raw_indices if positive_mask[i]]\n        if not positives:\n            continue\n        ordered = sorted(raw_indices, key=lambda i: predictions[i], reverse=True)\n        total += 1\n        if positive_mask[ordered[0]]:\n            hits += 1\n        best_positive_rank = min(ordered.index(i) + 1 for i in positives)\n        rank_fractions.append(best_positive_rank / max(1, len(ordered)))\n    mean_rank = sum(rank_fractions) / len(rank_fractions) if rank_fractions else 1.0\n    return hits / total if total else 0.0, mean_rank, total\n\n\ndef grouped_top1_rate_subset(rows, predictions, positive_mask, indices):\n    prediction_by_raw = {raw_index: predictions[local] for local, raw_index in enumerate(indices)}\n    grouped = defaultdict(list)\n    for raw_index in indices:\n        step = int(row_float(rows[raw_index], "step", 0.0))\n        grouped[(rows[raw_index].get("game_id", ""), step)].append(raw_index)\n\n    hits = 0\n    total = 0\n    rank_fractions = []\n    for raw_indices in grouped.values():\n        positives = [i for i in raw_indices if positive_mask[i]]\n        if not positives:\n            continue\n        ordered = sorted(raw_indices, key=lambda i: prediction_by_raw.get(i, 0.0), reverse=True)\n        total += 1\n        if positive_mask[ordered[0]]:\n            hits += 1\n        best_positive_rank = min(ordered.index(i) + 1 for i in positives)\n        rank_fractions.append(best_positive_rank / max(1, len(ordered)))\n    mean_rank = sum(rank_fractions) / len(rank_fractions) if rank_fractions else 1.0\n    return hits / total if total else 0.0, mean_rank, total\n\n\ndef tune_blend(rows, probabilities, positive_mask, indices, score_scale):\n    best = {"blend": 0.0, "top1": -1.0, "rank": 1.0}\n    heuristic_scores = [row_float(row, "heuristic_score_scaled", 0.0) * 100.0 for row in rows]\n    model_scores = [(probability - 0.5) * score_scale for probability in probabilities]\n    for step in range(0, 61):\n        blend = step / 100.0\n        combined = [\n            heuristic_scores[i] * (1.0 - blend) + (heuristic_scores[i] + model_scores[i]) * blend\n            for i in range(len(rows))\n        ]\n        top1, rank, _ = grouped_top1_rate(rows, combined, positive_mask, indices)\n        if top1 > best["top1"] or (abs(top1 - best["top1"]) < 1e-9 and rank < best["rank"]):\n            best = {"blend": blend, "top1": top1, "rank": rank}\n    return best\n\n\ndef save_member_checkpoint(model, best_state, nn, args, member_seed, epoch, best_valid_objective, checkpoint_context):\n    saved_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n    model.load_state_dict(best_state)\n    try:\n        linear_layers = [module for module in model.net if isinstance(module, nn.Linear)]\n        layers = []\n        for index, layer in enumerate(linear_layers):\n            layers.append(\n                {\n                    "weights": layer.weight.detach().cpu().tolist(),\n                    "bias": layer.bias.detach().cpu().tolist(),\n                    "activation": "relu" if index < len(linear_layers) - 1 else "linear",\n                }\n            )\n    finally:\n        model.load_state_dict(saved_state)\n    payload = {\n        "version": "v9",\n        "model_type": "mlp_relu_candidate_ranker",\n        "features": checkpoint_context["feature_names"],\n        "mean": dict(zip(checkpoint_context["feature_names"], checkpoint_context["means"])),\n        "scale": dict(zip(checkpoint_context["feature_names"], checkpoint_context["scales"])),\n        "layers": layers,\n        "activation": "relu",\n        "score_scale": args.score_scale,\n        "member_seed": member_seed,\n        "epoch": epoch,\n        "best_validation_objective": best_valid_objective,\n    }\n    checkpoint_path = checkpoint_context["checkpoint_dir"] / f"member_seed{member_seed}_epoch{epoch:03d}.json"\n    checkpoint_path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")\n    api = checkpoint_context.get("api")\n    if api is None:\n        print(f"Saved local v9 checkpoint {checkpoint_path}", flush=True)\n        return checkpoint_path\n    remote_path = f"{HF_REMOTE_PREFIX}/checkpoints/{checkpoint_path.name}"\n    try:\n        api.upload_file(\n            path_or_fileobj=str(checkpoint_path),\n            path_in_repo=remote_path,\n            repo_id=checkpoint_context["hf_repo_id"],\n            repo_type=checkpoint_context["hf_repo_type"],\n            commit_message=f"Upload v9 checkpoint member_seed={member_seed} epoch={epoch}",\n        )\n        print(f"Uploaded v9 checkpoint to {remote_path}", flush=True)\n    except Exception as exc:\n        print(f"Failed to upload v9 checkpoint {checkpoint_path.name}: {exc}", flush=True)\n    return checkpoint_path\n\n\ndef train_member(torch, nn, functional, args, member_seed, tensors, pairs, feature_count, eval_context, checkpoint_context):\n    torch.manual_seed(member_seed)\n    x_train, y_train, w_train, x_valid, y_valid, w_valid = tensors\n    device = x_train.device\n\n    class CandidateRanker(nn.Module):\n        def __init__(self, input_size):\n            super().__init__()\n            self.net = nn.Sequential(\n                nn.Linear(input_size, 160),\n                nn.ReLU(),\n                nn.Dropout(args.dropout),\n                nn.Linear(160, 80),\n                nn.ReLU(),\n                nn.Dropout(max(0.05, args.dropout * 0.70)),\n                nn.Linear(80, 40),\n                nn.ReLU(),\n                nn.Dropout(max(0.03, args.dropout * 0.45)),\n                nn.Linear(40, 1),\n            )\n\n        def forward(self, x):\n            return self.net(x)\n\n    model = CandidateRanker(feature_count).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    bce_loss = nn.BCEWithLogitsLoss(reduction="none")\n    train_pairs, valid_pairs = pairs\n    rows, valid_indices, positive_mask = eval_context\n    history = []\n    best_state = None\n    best_valid_objective = float("inf")\n    stale_epochs = 0\n    started = time.time()\n\n    def evaluate(x_tensor, y_tensor, w_tensor, pair_list):\n        model.eval()\n        with torch.no_grad():\n            logits = model(x_tensor)\n            probs = torch.sigmoid(logits)\n            weighted_bce = float((bce_loss(logits, y_tensor) * w_tensor).mean().detach().cpu())\n            mse = float(((probs - y_tensor) ** 2).mean().detach().cpu())\n            accuracy = float(((probs >= 0.5) == (y_tensor >= 0.5)).float().mean().detach().cpu())\n            pair_loss = 0.0\n            pair_accuracy = 0.0\n            if pair_list:\n                pos_idx = torch.tensor([p[0] for p in pair_list], dtype=torch.long, device=device)\n                neg_idx = torch.tensor([p[1] for p in pair_list], dtype=torch.long, device=device)\n                pair_w = torch.tensor([p[2] for p in pair_list], dtype=torch.float32, device=device)\n                margins = logits[pos_idx] - logits[neg_idx]\n                pair_loss = float((functional.softplus(-margins.view(-1)) * pair_w).mean().detach().cpu())\n                pair_accuracy = float((margins > 0).float().mean().detach().cpu())\n        return weighted_bce, mse, accuracy, pair_loss, pair_accuracy\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        permutation = torch.randperm(x_train.shape[0], device=device)\n        for start in range(0, x_train.shape[0], args.batch_size):\n            batch_idx = permutation[start:start + args.batch_size]\n            optimizer.zero_grad(set_to_none=True)\n            logits = model(x_train[batch_idx])\n            loss = (bce_loss(logits, y_train[batch_idx]) * w_train[batch_idx]).mean()\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer.step()\n\n        if train_pairs and args.pair_loss_weight > 0.0:\n            rng = random.Random(member_seed + epoch)\n            shuffled_pairs = train_pairs[:]\n            rng.shuffle(shuffled_pairs)\n            for start in range(0, len(shuffled_pairs), args.batch_size):\n                batch = shuffled_pairs[start:start + args.batch_size]\n                pos_idx = torch.tensor([p[0] for p in batch], dtype=torch.long, device=device)\n                neg_idx = torch.tensor([p[1] for p in batch], dtype=torch.long, device=device)\n                pair_w = torch.tensor([p[2] for p in batch], dtype=torch.float32, device=device)\n                optimizer.zero_grad(set_to_none=True)\n                margins = model(x_train[pos_idx]) - model(x_train[neg_idx])\n                pair_loss = (functional.softplus(-margins.view(-1)) * pair_w).mean()\n                loss = pair_loss * args.pair_loss_weight\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                optimizer.step()\n\n        train_bce, train_mse, train_acc, train_pair_loss, train_pair_acc = evaluate(x_train, y_train, w_train, train_pairs)\n        valid_bce, valid_mse, valid_acc, valid_pair_loss, valid_pair_acc = evaluate(x_valid, y_valid, w_valid, valid_pairs)\n        model.eval()\n        with torch.no_grad():\n            valid_probs = torch.sigmoid(model(x_valid)).view(-1).detach().cpu().tolist()\n        valid_top1, valid_rank, _ = grouped_top1_rate_subset(rows, valid_probs, positive_mask, valid_indices)\n        valid_objective = (\n            valid_bce\n            + args.pair_loss_weight * valid_pair_loss\n            + (1.0 - valid_top1) * 1.15\n            + valid_rank * 0.30\n        )\n        item = {\n            "epoch": epoch,\n            "train_logloss": train_bce,\n            "valid_logloss": valid_bce,\n            "train_pair_loss": train_pair_loss,\n            "valid_pair_loss": valid_pair_loss,\n            "train_pair_accuracy": train_pair_acc,\n            "valid_pair_accuracy": valid_pair_acc,\n            "valid_turn_top1_positive_rate": valid_top1,\n            "valid_positive_mean_rank_fraction": valid_rank,\n            "train_accuracy": train_acc,\n            "valid_accuracy": valid_acc,\n            "train_mse": train_mse,\n            "valid_mse": valid_mse,\n            "valid_objective": valid_objective,\n            "elapsed_seconds": round(time.time() - started, 2),\n        }\n        history.append(item)\n        print(\n            f"member_seed={member_seed} epoch={epoch:03d}/{args.epochs} "\n            f"valid_bce={valid_bce:.5f} valid_pair_acc={valid_pair_acc:.4f} "\n            f"valid_top1={valid_top1:.4f} valid_obj={valid_objective:.5f} "\n            f"elapsed={item[\'elapsed_seconds\']:.1f}s",\n            flush=True,\n        )\n        if valid_objective + 1e-5 < best_valid_objective:\n            best_valid_objective = valid_objective\n            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n            stale_epochs = 0\n        else:\n            stale_epochs += 1\n            if stale_epochs >= args.patience:\n                print(f"member_seed={member_seed} early_stop epoch={epoch} best_valid_objective={best_valid_objective:.5f}", flush=True)\n                break\n\n        if (\n            checkpoint_context is not None\n            and args.checkpoint_every > 0\n            and epoch % args.checkpoint_every == 0\n            and best_state is not None\n        ):\n            save_member_checkpoint(\n                model, best_state, nn, args, member_seed, epoch, best_valid_objective, checkpoint_context\n            )\n\n    if best_state is not None:\n        model.load_state_dict(best_state)\n    model.eval()\n    linear_layers = [module for module in model.net if isinstance(module, nn.Linear)]\n    layers = []\n    for index, layer in enumerate(linear_layers):\n        layers.append(\n            {\n                "weights": layer.weight.detach().cpu().tolist(),\n                "bias": layer.bias.detach().cpu().tolist(),\n                "activation": "relu" if index < len(linear_layers) - 1 else "linear",\n            }\n        )\n    return model, layers, history, best_valid_objective\n\n\ndef train(args):\n    try:\n        import torch\n        from torch import nn\n        import torch.nn.functional as functional\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("PyTorch is required for v9 training. Install torch or run on Kaggle/Colab.") from exc\n\n    random.seed(args.seed)\n    data_path = find_training_csv(args.csv, getattr(args, "prefer_local_data", False))\n    rows, feature_names, x_raw, y, sample_weights, selected, counterfactual = read_rows(data_path)\n    train_indices, valid_indices, games, valid_games = split_by_game(rows, args.seed)\n    means, scales, normalize = normalize_from_train(x_raw, train_indices)\n    all_x = normalize(x_raw)\n    train_x = [all_x[i] for i in train_indices]\n    valid_x = [all_x[i] for i in valid_indices]\n    train_y = [y[i] for i in train_indices]\n    valid_y = [y[i] for i in valid_indices]\n    train_w = [sample_weights[i] for i in train_indices]\n    valid_w = [sample_weights[i] for i in valid_indices]\n\n    train_local = {raw_index: local for local, raw_index in enumerate(train_indices)}\n    valid_local = {raw_index: local for local, raw_index in enumerate(valid_indices)}\n    train_pairs = build_pair_indices(rows, y, selected, counterfactual, train_indices, train_local, args.max_pairs_per_turn)\n    valid_pairs = build_pair_indices(rows, y, selected, counterfactual, valid_indices, valid_local, args.max_pairs_per_turn)\n    positive_mask = [selected[i] or counterfactual[i] or y[i] >= 0.55 for i in range(len(rows))]\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    x_train = torch.tensor(train_x, dtype=torch.float32, device=device)\n    y_train = torch.tensor(train_y, dtype=torch.float32, device=device).view(-1, 1)\n    w_train = torch.tensor(train_w, dtype=torch.float32, device=device).view(-1, 1)\n    x_valid = torch.tensor(valid_x, dtype=torch.float32, device=device)\n    y_valid = torch.tensor(valid_y, dtype=torch.float32, device=device).view(-1, 1)\n    w_valid = torch.tensor(valid_w, dtype=torch.float32, device=device).view(-1, 1)\n    tensors = (x_train, y_train, w_train, x_valid, y_valid, w_valid)\n    pairs = (train_pairs, valid_pairs)\n    eval_context = (rows, valid_indices, positive_mask)\n\n    print(json.dumps({\n        "csv": str(data_path),\n        "rows": len(rows),\n        "features": len(feature_names),\n        "games": len(games),\n        "validation_games": len(valid_games),\n        "train_rows": len(train_x),\n        "validation_rows": len(valid_x),\n        "train_pairs": len(train_pairs),\n        "validation_pairs": len(valid_pairs),\n        "positive_rate": sum(1 for value in y if value >= 0.5) / len(y),\n        "selected_rate": sum(1 for value in selected if value) / len(selected),\n        "counterfactual_rate": sum(1 for value in counterfactual if value) / len(counterfactual),\n        "ensemble_size": args.ensemble_size,\n        "device": str(device),\n    }, indent=2), flush=True)\n\n    export_dir = Path(args.export_dir)\n    graph_dir = export_dir / "graphs"\n    checkpoint_dir = export_dir / "checkpoints"\n    export_dir.mkdir(parents=True, exist_ok=True)\n    graph_dir.mkdir(parents=True, exist_ok=True)\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n\n    hf_api = None\n    if args.upload:\n        load_dotenv()\n        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n        if not token:\n            raise RuntimeError("HF_TOKEN is required for --upload.")\n        try:\n            from huggingface_hub import HfApi\n        except ModuleNotFoundError as exc:\n            raise RuntimeError("Install huggingface_hub to upload: pip install huggingface_hub") from exc\n        hf_api = HfApi(token=token)\n        hf_api.create_repo(repo_id=args.hf_repo_id, repo_type=args.hf_repo_type, exist_ok=True)\n\n    checkpoint_context = {\n        "feature_names": feature_names,\n        "means": means,\n        "scales": scales,\n        "checkpoint_dir": checkpoint_dir,\n        "api": hf_api,\n        "hf_repo_id": args.hf_repo_id,\n        "hf_repo_type": args.hf_repo_type,\n    }\n\n    members = []\n    histories = []\n    models = []\n    for member_index in range(args.ensemble_size):\n        member_seed = args.seed + member_index * 9973\n        model, layers, history, best_objective = train_member(\n            torch,\n            nn,\n            functional,\n            args,\n            member_seed,\n            tensors,\n            pairs,\n            len(feature_names),\n            eval_context,\n            checkpoint_context,\n        )\n        models.append(model)\n        histories.append({"seed": member_seed, "history": history, "best_validation_objective": best_objective})\n        members.append(\n            {\n                "version": "v9",\n                "model_type": "mlp_relu_candidate_ranker",\n                "features": feature_names,\n                "mean": dict(zip(feature_names, means)),\n                "scale": dict(zip(feature_names, scales)),\n                "layers": layers,\n                "activation": "relu",\n                "score_scale": args.score_scale,\n            }\n        )\n\n    for model in models:\n        model.eval()\n    with torch.no_grad():\n        all_tensor = torch.tensor(all_x, dtype=torch.float32, device=device)\n        member_probs = []\n        for model in models:\n            logits = model(all_tensor).view(-1).detach().cpu().tolist()\n            member_probs.append([sigmoid_prob(value) for value in logits])\n    all_probs = [sum(member[i] for member in member_probs) / len(member_probs) for i in range(len(rows))]\n    train_predictions = [all_probs[i] for i in train_indices]\n    valid_predictions = [all_probs[i] for i in valid_indices]\n    valid_top1, valid_rank, valid_turns = grouped_top1_rate(rows, all_probs, positive_mask, valid_indices)\n    train_top1, train_rank, train_turns = grouped_top1_rate(rows, all_probs, positive_mask, train_indices)\n    tuned_blend = tune_blend(rows, all_probs, positive_mask, valid_indices, args.score_scale)\n\n    def log_loss(preds, labels):\n        return -sum(\n            label * math.log(max(1e-9, pred)) + (1.0 - label) * math.log(max(1e-9, 1.0 - pred))\n            for pred, label in zip(preds, labels)\n        ) / len(labels)\n\n    def accuracy(preds, labels):\n        return sum((pred >= 0.5) == (label >= 0.5) for pred, label in zip(preds, labels)) / len(labels)\n\n    metrics = {\n        "rows": len(rows),\n        "features": len(feature_names),\n        "games": len(games),\n        "train_rows": len(train_x),\n        "validation_rows": len(valid_x),\n        "train_pairs": len(train_pairs),\n        "validation_pairs": len(valid_pairs),\n        "positive_rate": sum(1 for value in y if value >= 0.5) / len(y),\n        "selected_rate": sum(1 for value in selected if value) / len(selected),\n        "counterfactual_rate": sum(1 for value in counterfactual if value) / len(counterfactual),\n        "train_accuracy": accuracy(train_predictions, train_y),\n        "validation_accuracy": accuracy(valid_predictions, valid_y),\n        "train_logloss": log_loss(train_predictions, train_y),\n        "validation_logloss": log_loss(valid_predictions, valid_y),\n        "train_turn_top1_positive_rate": train_top1,\n        "validation_turn_top1_positive_rate": valid_top1,\n        "train_positive_mean_rank_fraction": train_rank,\n        "validation_positive_mean_rank_fraction": valid_rank,\n        "train_turns_ranked": train_turns,\n        "validation_turns_ranked": valid_turns,\n        "tuned_blend": tuned_blend["blend"],\n        "tuned_blend_validation_top1": tuned_blend["top1"],\n        "tuned_blend_validation_rank_fraction": tuned_blend["rank"],\n        "ensemble_size": len(members),\n        "device": str(device),\n        "pair_loss_weight": args.pair_loss_weight,\n        "dropout": args.dropout,\n        "score_scale": args.score_scale,\n    }\n\n    artifact = {\n        "version": "v9",\n        "created_at": int(time.time()),\n        "source_csv": str(data_path),\n        "model_type": "ensemble_mlp_relu_candidate_ranker",\n        "members": members,\n        "blend": tuned_blend["blend"],\n        "metrics": metrics,\n    }\n\n    (export_dir / "model_weights_v9.json").write_text(json.dumps(artifact, indent=2, sort_keys=True), encoding="utf-8")\n    (export_dir / "feature_schema_v9.json").write_text(json.dumps({"features": feature_names}, indent=2), encoding="utf-8")\n    (export_dir / "metrics_v9.json").write_text(json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8")\n    (export_dir / "training_history_v9.json").write_text(json.dumps(histories, indent=2, sort_keys=True), encoding="utf-8")\n    with (export_dir / "predictions_v9.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.writer(f)\n        writer.writerow(["row_index", "label", "prediction", "selected", "counterfactual_positive", "game_result", "split"])\n        valid_set = set(valid_indices)\n        for i, pred in enumerate(all_probs):\n            writer.writerow([\n                i,\n                y[i],\n                pred,\n                float(selected[i]),\n                float(counterfactual[i]),\n                rows[i].get("game_result", 0.0),\n                "validation" if i in valid_set else "train",\n            ])\n\n    try:\n        import matplotlib.pyplot as plt\n\n        first_history = histories[0]["history"] if histories else []\n        epochs = [item["epoch"] for item in first_history]\n        plt.figure(figsize=(7, 4))\n        plt.plot(epochs, [item["valid_logloss"] for item in first_history], label="validation BCE")\n        plt.plot(epochs, [item["valid_objective"] for item in first_history], label="validation objective")\n        plt.xlabel("epoch")\n        plt.ylabel("loss")\n        plt.title("v9 first member validation loss")\n        plt.legend()\n        plt.tight_layout()\n        plt.savefig(graph_dir / "loss_curve_v9.png", dpi=150)\n        plt.close()\n\n        plt.figure(figsize=(7, 4))\n        plt.hist([pred for pred, label in zip(all_probs, y) if label >= 0.5], bins=30, alpha=0.65, label="positive")\n        plt.hist([pred for pred, label in zip(all_probs, y) if label < 0.5], bins=30, alpha=0.65, label="negative")\n        plt.xlabel("prediction")\n        plt.ylabel("rows")\n        plt.title("v9 ensemble prediction distribution")\n        plt.legend()\n        plt.tight_layout()\n        plt.savefig(graph_dir / "prediction_histogram_v9.png", dpi=150)\n        plt.close()\n    except ModuleNotFoundError:\n        print("matplotlib is not installed; skipped graph generation.", flush=True)\n\n    print(json.dumps(metrics, indent=2, sort_keys=True), flush=True)\n    print(f"Saved v9 model artifact: {export_dir / \'model_weights_v9.json\'}", flush=True)\n\n    if args.upload and hf_api is not None:\n        hf_api.upload_folder(\n            folder_path=str(export_dir),\n            repo_id=args.hf_repo_id,\n            repo_type=args.hf_repo_type,\n            path_in_repo=HF_REMOTE_PREFIX,\n            commit_message="Upload v9 Orbit Wars supervised ensemble ranker artifacts and graphs",\n        )\n        print(f"Uploaded {export_dir} to https://huggingface.co/{args.hf_repo_id}/tree/main/{HF_REMOTE_PREFIX}", flush=True)\n\n\ndef main():\n    train(parse_args())\n\n\nif __name__ == "__main__":\n    main()\n'

v9_ranker = types.ModuleType("inline_v9_ranker")
v9_ranker.__file__ = "<inline_v9_ranker>"
sys.modules["inline_v9_ranker"] = v9_ranker
exec(V9_RANKER_CODE, v9_ranker.__dict__)
print("Loaded inline v9 ranker; no companion .py file is required in the notebook runtime.")


In [ ]:
import argparse
import os

# Defaults are still environment-overridable from notebook settings.
defaults = {
    "V9_EPOCHS": "280",
    "V9_BATCH_SIZE": "4096",
    "V9_LR": "0.00045",
    "V9_WEIGHT_DECAY": "0.00030",
    "V9_PAIR_LOSS_WEIGHT": "1.05",
    "V9_MAX_PAIRS_PER_TURN": "12",
    "V9_ENSEMBLE_SIZE": "8",
    "V9_PATIENCE": "36",
    "V9_DROPOUT": "0.13",
    "V9_SCORE_SCALE": "205.0",
    "V9_CHECKPOINT_EVERY": "40",
}
for key, value in defaults.items():
    os.environ.setdefault(key, value)

export_dir = repo_root / "notebooks" / "v9" / "exports"
args = argparse.Namespace(
    csv=os.environ.get("CANDIDATES_CSV", ""),
    prefer_local_data=os.environ.get("V9_PREFER_LOCAL_DATA", "0") == "1",
    export_dir=str(export_dir),
    epochs=int(os.environ["V9_EPOCHS"]),
    batch_size=int(os.environ["V9_BATCH_SIZE"]),
    lr=float(os.environ["V9_LR"]),
    weight_decay=float(os.environ["V9_WEIGHT_DECAY"]),
    pair_loss_weight=float(os.environ["V9_PAIR_LOSS_WEIGHT"]),
    max_pairs_per_turn=int(os.environ["V9_MAX_PAIRS_PER_TURN"]),
    ensemble_size=int(os.environ["V9_ENSEMBLE_SIZE"]),
    patience=int(os.environ["V9_PATIENCE"]),
    dropout=float(os.environ["V9_DROPOUT"]),
    score_scale=float(os.environ["V9_SCORE_SCALE"]),
    seed=int(os.environ.get("V9_SEED", "907")),
    checkpoint_every=int(os.environ["V9_CHECKPOINT_EVERY"]),
    upload=os.environ.get("V9_UPLOAD", "1") != "0",
    hf_repo_id=os.environ.get("HF_REPO_ID", v9_ranker.HF_REPO_ID),
    hf_repo_type=os.environ.get("HF_REPO_TYPE", v9_ranker.HF_REPO_TYPE),
)
print({
    "epochs": args.epochs,
    "batch_size": args.batch_size,
    "ensemble_size": args.ensemble_size,
    "pair_loss_weight": args.pair_loss_weight,
    "dropout": args.dropout,
    "checkpoint_every": args.checkpoint_every,
    "upload": args.upload,
    "prefer_local_data": args.prefer_local_data,
    "export_dir": args.export_dir,
})
v9_ranker.train(args)


In [ ]:
import json
from pathlib import Path

export_dir = repo_root / "notebooks" / "v9" / "exports"
metrics_path = export_dir / "metrics_v9.json"
weights_path = export_dir / "model_weights_v9.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, sort_keys=True))
print("Model:", weights_path)
print("Graphs:", export_dir / "graphs")
